# nb07b — FCOS Tuning (F1) + Comparison + Submission

**Purpose.** Warm-start from F0 (in `fcos-1` dataset), train F1 with a tighter center sampling radius (1.0 vs 1.5), evaluate both, generate submission from the winner.

**Config (F1 tuning):**
- `center_sampling_radius = 1.0` (down from 1.5)
- `iou_loss_type = "giou"` (unchanged)

**Artifacts produced:**
- `fcos_F1.pth`, `val_preds_fcos_F1.json`, `metrics_fcos_F1.json`, `speed_fcos_F1.json`, `train_log_fcos_F1.json`
- `submission_fcos.csv` — from whichever of F0/F1 wins on val mAP
- W&B run `fcos_F1`

In [1]:
import sys, json, time, shutil, os
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision.ops import box_iou
from torchvision.models.detection import fcos_resnet50_fpn
from torchvision.models.detection.fcos import FCOSClassificationHead
from torchvision.transforms import functional as TF

# ---- Paths -----------------------------------------------------------------
COMP_DIR    = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
DATASET_DIR = Path("/kaggle/input/datasets/suchok/my-dataset")
OUTPUTS_DIR = Path("/kaggle/input/datasets/suchok/outputs")
FCOS1_DIR   = Path("/kaggle/input/datasets/suchok/fcos-1")           # ← adjust if path differs
WORK_DIR    = Path("/kaggle/working")

assert COMP_DIR.exists(),    f"missing competition: {COMP_DIR}"
assert DATASET_DIR.exists(), f"missing my-dataset: {DATASET_DIR}"
assert OUTPUTS_DIR.exists(), f"missing outputs: {OUTPUTS_DIR}"
assert FCOS1_DIR.exists(),   f"missing fcos-1: {FCOS1_DIR} — add as dataset input"
WORK_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(DATASET_DIR))
import common, importlib
importlib.reload(common)
from common import (
    set_seeds, SEED, CLASS_NAMES, NUM_CLASSES,
    load_splits, index_split, filter_by_clips,
    VisDroneDetectionDataset, detection_collate,
    build_fullval_gt,
    save_checkpoint, save_metrics,
    IMAGENET_MEAN, IMAGENET_STD,
)

set_seeds(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| torch:", torch.__version__)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

chosen = json.load(open(OUTPUTS_DIR / "chosen_preprocessing.json"))
STRAT  = chosen["strategy"]
SIZE   = int(chosen["size"])
print("Locked preprocessing:", STRAT, SIZE)

train_clips, val_clips = load_splits(OUTPUTS_DIR / "splits")
train_index = filter_by_clips(index_split(COMP_DIR / "train"), train_clips)
val_index   = filter_by_clips(index_split(COMP_DIR / "val"),   val_clips)
print(f"train frames: {len(train_index):,} | val frames: {len(val_index):,}")

TEST_CSV = COMP_DIR / "test_images.csv"

# ---- Restore F0 artifacts + GT from fcos-1 --------------------------------
for name in ["fcos_F0.pth", "metrics_fcos_F0.json",
             "speed_fcos_F0.json", "val_preds_fcos_F0.json",
             "gt_fullval_ignore.json"]:
    src = FCOS1_DIR / name
    dst = WORK_DIR / name
    if src.exists():
        shutil.copy(src, dst)
        print(f"restored: {name}")
    else:
        print(f"⚠️  missing in fcos-1: {name}")

gt_fullval_ignore = WORK_DIR / "gt_fullval_ignore.json"
F0_CKPT = WORK_DIR / "fcos_F0.pth"
assert gt_fullval_ignore.exists(), "missing GT"
assert F0_CKPT.exists(),           "missing F0 checkpoint"

with open(WORK_DIR / "metrics_fcos_F0.json") as f:
    metrics_F0 = json.load(f)
with open(WORK_DIR / "speed_fcos_F0.json") as f:
    speed_F0 = json.load(f)
print("F0 metrics:", metrics_F0)

DEVICE: cuda | torch: 2.11.0+cu130
Locked preprocessing: letterbox 1280
train frames: 24,201 | val frames: 2,846
restored: fcos_F0.pth
restored: metrics_fcos_F0.json
restored: speed_fcos_F0.json
restored: val_preds_fcos_F0.json
restored: gt_fullval_ignore.json
F0 metrics: {'mAP': 0.02575235259516088, 'AP50': 0.0502928573932851, 'AP75': 0.025184789379852572, 'AP_small': 0.008857352698771753, 'AP_medium': 0.020156772111203616, 'AP_large': 0.05931758091227115}


In [2]:
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
import wandb
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
WANDB_PROJECT = "visdrone-det"
print("W&B ready. project:", WANDB_PROJECT)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: suchok-pn (suchok-pn-ukrainian-catholic-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready. project: visdrone-det


In [3]:
def build_fcos(center_sampling_radius=1.5, iou_loss_type="giou",
               score_thresh=0.05, nms_thresh=0.5):
    m = fcos_resnet50_fpn(
        weights=None, weights_backbone=None,
        min_size=SIZE, max_size=SIZE,
        score_thresh=score_thresh, nms_thresh=nms_thresh,
        detections_per_img=500,
        center_sampling_radius=center_sampling_radius,
    )
    in_channels = m.head.classification_head.cls_logits.in_channels
    num_anchors = m.head.classification_head.num_anchors
    m.head.classification_head = FCOSClassificationHead(
        in_channels=in_channels, num_anchors=num_anchors,
        num_classes=NUM_CLASSES + 1,
        norm_layer=lambda c: torch.nn.GroupNorm(32, c),
    )
    m.iou_loss_type = iou_loss_type
    return m


F1_CFG = {"center_sampling_radius": 1.0, "iou_loss_type": "giou"}
model = build_fcos(**F1_CFG).to(DEVICE)
print(f"F1 config: {F1_CFG}")

# Warm-start from F0
state = torch.load(F0_CKPT, map_location=DEVICE)
model.load_state_dict(state, strict=True)
print(f"warm-started from {F0_CKPT.name}")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {n_params/1e6:.2f}M")

F1 config: {'center_sampling_radius': 1.0, 'iou_loss_type': 'giou'}
warm-started from fcos_F0.pth
trainable params: 32.13M


In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    VisDroneDetectionDataset(train_index, STRAT, SIZE, is_train=False),
    batch_size=2, shuffle=True, num_workers=2,
    collate_fn=detection_collate, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    VisDroneDetectionDataset(val_index, STRAT, SIZE, is_train=False),
    batch_size=2, shuffle=False, num_workers=2,
    collate_fn=detection_collate, pin_memory=True,
)
print(f"train batches: {len(train_loader):,} | val batches: {len(val_loader):,}")

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=5e-3, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

model.train()
imgs, tgts = next(iter(train_loader))
imgs = [i.to(DEVICE) for i in imgs]
tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
with torch.no_grad():
    ld = model(imgs, tgts)
print("smoke loss:", {k: round(float(v.detach()), 4) for k, v in ld.items()})

train batches: 12,100 | val batches: 1,423
smoke loss: {'classification': 0.4071, 'bbox_regression': 0.3557, 'bbox_ctrness': 0.6543}


In [5]:
def train_one_epoch(model, loader, optimizer, scheduler,
                     device, run_name, wandb_run,
                     log_every=100, wandb_log_every=20):
    model.train()
    running = {"loss": 0.0, "loss_cls": 0.0, "loss_box": 0.0, "loss_centerness": 0.0}
    n_batches = 0
    t_start = time.time()

    for step, (images, targets) in enumerate(tqdm(loader, desc=f"{run_name}")):
        images  = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()}
                   for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())

        optimizer.zero_grad(set_to_none=True)
        losses.backward()
        optimizer.step()

        zero = torch.tensor(0.0, device=device)
        cls_loss = loss_dict.get("classification", zero)
        box_loss = loss_dict.get("bbox_regression", zero)
        ctr_loss = loss_dict.get("bbox_ctrness", zero)

        running["loss"]           += float(losses.detach())
        running["loss_cls"]       += float(cls_loss.detach())
        running["loss_box"]       += float(box_loss.detach())
        running["loss_centerness"] += float(ctr_loss.detach())
        n_batches += 1

        if step % log_every == 0 and step > 0:
            elapsed = time.time() - t_start
            spb = elapsed / (step + 1)
            remaining = (len(loader) - step - 1) * spb
            print(f"[{run_name} {step}/{len(loader)}] "
                  f"elapsed={elapsed/60:.1f}min {spb:.2f}s/batch "
                  f"ETA={remaining/3600:.2f}h loss={float(losses.detach()):.4f}",
                  flush=True)

        if step % wandb_log_every == 0:
            wandb_run.log({
                "train/loss":           float(losses.detach()),
                "train/loss_cls":       float(cls_loss.detach()),
                "train/loss_box":       float(box_loss.detach()),
                "train/loss_centerness": float(ctr_loss.detach()),
                "train/step":           step,
            })

    scheduler.step()
    mean = {k: v / max(1, n_batches) for k, v in running.items()}
    return mean, time.time() - t_start

In [6]:
@torch.no_grad()
def run_detector_eval(detector, image_rgb, score_thr=0.001, device=None):
    if device is None:
        device = next(detector.parameters()).device
    detector.eval()
    tensor = TF.to_tensor(image_rgb)
    tensor = TF.normalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
    out = detector([tensor.to(device)])[0]
    keep = out["scores"] >= score_thr
    return (out["boxes"][keep].cpu().numpy(),
            out["labels"][keep].cpu().numpy(),
            out["scores"][keep].cpu().numpy())


def predict_with_strategy_eval(detector, image_path, strategy, size,
                                score_thr=0.001, device=None):
    from common import apply_strategy, boxes_to_original
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    h0, w0 = img.shape[:2]
    inp, meta = apply_strategy(img, strategy, size, seed_key=image_path)
    t0 = time.time()
    boxes, labels, scores = run_detector_eval(detector, inp,
                                               score_thr=score_thr, device=device)
    dt = time.time() - t0
    boxes_orig = boxes_to_original(boxes, meta, strategy, w0, h0)
    return boxes_orig, labels, scores, dt, meta


def predictions_to_coco_eval(df_index, detector, strategy, size, out_json,
                              score_thr=0.001, device=None):
    preds = []
    id_map = {row["image_path"]: i for i, (_, row) in enumerate(
        df_index.iterrows(), start=1)}
    valid_cat_ids = set(range(1, NUM_CLASSES + 1))
    for _, row in tqdm(df_index.iterrows(), total=len(df_index),
                       desc=f"pred {strategy}-{size}"):
        boxes, labels, scores, _, _ = predict_with_strategy_eval(
            detector, row["image_path"], strategy, size,
            score_thr=score_thr, device=device)
        img_id = id_map[row["image_path"]]
        for (x1, y1, x2, y2), l, s in zip(boxes, labels, scores):
            if int(l) not in valid_cat_ids:
                continue
            preds.append({
                "image_id": int(img_id),
                "category_id": int(l),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "score": float(s),
            })
    with open(out_json, "w") as f:
        json.dump(preds, f)
    print(f"wrote {len(preds):,} predictions -> {out_json}")


def coco_eval_summary_safe(gt_json, dt_json):
    import contextlib, io
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO(gt_json)
        if os.path.getsize(dt_json) == 0 or json.load(open(dt_json)) == []:
            return {k: 0.0 for k in ["mAP", "AP50", "AP75",
                                     "AP_small", "AP_medium", "AP_large"]}
        coco_dt = coco_gt.loadRes(dt_json)
        e = COCOeval(coco_gt, coco_dt, "bbox")
        e.evaluate(); e.accumulate(); e.summarize()
    return {
        "mAP":       float(e.stats[0]),
        "AP50":      float(e.stats[1]),
        "AP75":      float(e.stats[2]),
        "AP_small":  float(e.stats[3]),
        "AP_medium": float(e.stats[4]),
        "AP_large":  float(e.stats[5]),
    }

In [7]:
run = wandb.init(
    project=WANDB_PROJECT, name="fcos_F1",
    config=dict(model="fcos_resnet50_fpn", variant="FCOS_F1",
                strategy=STRAT, size=SIZE, epochs=1, lr=5e-3, batch_size=2,
                **F1_CFG, seed=SEED),
    reinit=True,
)

t0 = time.time()
train_stats, train_seconds = train_one_epoch(
    model, train_loader, optimizer, scheduler, DEVICE,
    run_name="F1", wandb_run=run, log_every=100, wandb_log_every=20,
)
print(f"\ntrain stats: {train_stats}")
print(f"train wall time: {train_seconds/3600:.2f} h")

save_checkpoint(model, WORK_DIR / "fcos_F1.pth")
with open(WORK_DIR / "train_log_fcos_F1.json", "w") as f:
    json.dump({"epoch": 1, "run": "F1",
               "mean_loss":     train_stats["loss"],
               "mean_loss_cls": train_stats["loss_cls"],
               "mean_loss_box": train_stats["loss_box"],
               "mean_loss_centerness": train_stats["loss_centerness"],
               "wall_seconds":  train_seconds,
               "config":        F1_CFG}, f, indent=2)
wandb.log({"train/epoch_loss": train_stats["loss"],
           "train/epoch_time_h": train_seconds/3600})
print("checkpoint + train_log saved")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261108_085254-fx096sj7
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fcos_F1
wandb: ⭐️ View project at https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det
wandb: 🚀 View run at https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det/runs/fx096sj7


F1:   0%|          | 0/12100 [00:00<?, ?it/s]

[F1 100/12100] elapsed=2.6min 1.57s/batch ETA=5.24h loss=1.4332
[F1 200/12100] elapsed=5.0min 1.50s/batch ETA=4.97h loss=1.3766
[F1 300/12100] elapsed=7.4min 1.48s/batch ETA=4.84h loss=1.3579
[F1 400/12100] elapsed=9.8min 1.47s/batch ETA=4.76h loss=1.4520
[F1 500/12100] elapsed=12.2min 1.46s/batch ETA=4.70h loss=1.4437
[F1 600/12100] elapsed=14.6min 1.45s/batch ETA=4.65h loss=1.3243
[F1 700/12100] elapsed=16.9min 1.45s/batch ETA=4.59h loss=1.3527
[F1 800/12100] elapsed=19.3min 1.45s/batch ETA=4.54h loss=1.2958
[F1 900/12100] elapsed=21.7min 1.45s/batch ETA=4.50h loss=1.3056
[F1 1000/12100] elapsed=24.1min 1.44s/batch ETA=4.45h loss=1.4711
[F1 1100/12100] elapsed=26.5min 1.44s/batch ETA=4.41h loss=1.3558
[F1 1200/12100] elapsed=28.9min 1.44s/batch ETA=4.37h loss=1.3121
[F1 1300/12100] elapsed=31.3min 1.44s/batch ETA=4.33h loss=1.3489
[F1 1400/12100] elapsed=33.7min 1.44s/batch ETA=4.28h loss=1.4261
[F1 1500/12100] elapsed=36.0min 1.44s/batch ETA=4.24h loss=1.2254
[F1 1600/12100] elapsed

In [8]:
torch.cuda.empty_cache()
model.eval()
preds_path = WORK_DIR / "val_preds_fcos_F1.json"
t0 = time.time()
predictions_to_coco_eval(val_index, model, STRAT, SIZE, str(preds_path),
                         score_thr=0.001, device=DEVICE)
print(f"inference: {(time.time()-t0)/60:.2f} min")

metrics_F1 = coco_eval_summary_safe(str(gt_fullval_ignore), str(preds_path))
print("=" * 60)
print("F1 — full val:")
for k, v in metrics_F1.items():
    print(f"  {k:10s}: {v:.4f}")
save_metrics(metrics_F1, WORK_DIR / "metrics_fcos_F1.json")
wandb.log({f"val/{k}": v for k, v in metrics_F1.items()})

pred letterbox-1280:   0%|          | 0/2846 [00:00<?, ?it/s]

wrote 1,393,770 predictions -> /kaggle/working/val_preds_fcos_F1.json
inference: 17.42 min
F1 — full val:
  mAP       : 0.0396
  AP50      : 0.0837
  AP75      : 0.0336
  AP_small  : 0.0201
  AP_medium : 0.0336
  AP_large  : 0.0853


In [9]:
model.eval()
if DEVICE == "cuda":
    torch.cuda.synchronize()
for i in range(5):
    _ = predict_with_strategy_eval(model, val_index.iloc[i]["image_path"],
                                    STRAT, SIZE, score_thr=0.5, device=DEVICE)
times = []
for i in range(50):
    row = val_index.iloc[(5 + i) % len(val_index)]
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    _ = predict_with_strategy_eval(model, row["image_path"], STRAT, SIZE,
                                    score_thr=0.5, device=DEVICE)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    times.append(time.time() - t0)
mean_s = float(np.mean(times))
speed_F1 = {"device": str(DEVICE),
            "gpu_ms_per_image": mean_s * 1000,
            "gpu_fps":          1.0 / mean_s,
            "n_images": 50, "score_thr": 0.5}
print("speed:", json.dumps(speed_F1, indent=2))
with open(WORK_DIR / "speed_fcos_F1.json", "w") as f:
    json.dump(speed_F1, f, indent=2)
wandb.log({"speed/gpu_fps": speed_F1["gpu_fps"],
           "speed/gpu_ms":  speed_F1["gpu_ms_per_image"]})

speed: {
  "device": "cuda",
  "gpu_ms_per_image": 298.8983106613159,
  "gpu_fps": 3.3456194442433906,
  "n_images": 50,
  "score_thr": 0.5
}


In [10]:
print("=" * 70)
print("F0 vs F1 comparison")
print("=" * 70)
print(f"{'Metric':<12} {'F0 (r=1.5)':<15} {'F1 (r=1.0)':<15} {'Δ':<10}")
for k in ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]:
    a, b = metrics_F0[k], metrics_F1[k]
    print(f"{k:<12} {a:<15.4f} {b:<15.4f} {b-a:+.4f}")
print()
print(f"F0 FPS: {speed_F0['gpu_fps']:.2f} | F1 FPS: {speed_F1['gpu_fps']:.2f}")

# Pick winner by mAP
if metrics_F1["mAP"] >= metrics_F0["mAP"]:
    winner = "F1"
    winner_ckpt = WORK_DIR / "fcos_F1.pth"
    winner_metrics = metrics_F1
    print(f"\n✅ WINNER: F1 (mAP {metrics_F1['mAP']:.4f} >= {metrics_F0['mAP']:.4f})")
else:
    winner = "F0"
    winner_ckpt = WORK_DIR / "fcos_F0.pth"
    winner_metrics = metrics_F0
    print(f"\n✅ WINNER: F0 (mAP {metrics_F0['mAP']:.4f} > {metrics_F1['mAP']:.4f})")

print(f"loading winner checkpoint for submission: {winner_ckpt.name}")
winner_model = build_fcos(**({"center_sampling_radius": 1.0} if winner == "F1"
                              else {"center_sampling_radius": 1.5})).to(DEVICE)
winner_model.load_state_dict(torch.load(winner_ckpt, map_location=DEVICE))
winner_model.eval()

F0 vs F1 comparison
Metric       F0 (r=1.5)      F1 (r=1.0)      Δ         
mAP          0.0258          0.0396          +0.0138
AP50         0.0503          0.0837          +0.0334
AP75         0.0252          0.0336          +0.0084
AP_small     0.0089          0.0201          +0.0113
AP_medium    0.0202          0.0336          +0.0135
AP_large     0.0593          0.0853          +0.0260

F0 FPS: 3.65 | F1 FPS: 3.35

✅ WINNER: F1 (mAP 0.0396 >= 0.0258)
loading winner checkpoint for submission: fcos_F1.pth


FCOS(
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (

In [11]:
test_df = pd.read_csv(TEST_CSV)
print(f"test frames: {len(test_df):,}")

submission_rows = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="test inference"):
    fname = row["file_name"]
    img_path = str(COMP_DIR / fname) if not os.path.isabs(fname) else fname
    boxes, labels, scores, _, _ = predict_with_strategy_eval(
        winner_model, img_path, STRAT, SIZE, score_thr=0.05, device=DEVICE)

    if len(scores) > 0:
        order = np.argsort(-scores)[:500]
        boxes, labels, scores = boxes[order], labels[order], scores[order]

    parts = []
    for (x1, y1, x2, y2), l, s in zip(boxes, labels, scores):
        cls_id = int(l) - 1
        if cls_id < 0 or cls_id >= NUM_CLASSES:
            continue
        w, h = float(x2 - x1), float(y2 - y1)
        if w <= 0 or h <= 0:
            continue
        parts.append(f"{cls_id} {float(s):.4f} {x1:.1f} {y1:.1f} {w:.1f} {h:.1f}")

    submission_rows.append({
        "image_id": row["image_id"],
        "PredictionString": " ".join(parts),
    })

sub_df = pd.DataFrame(submission_rows)
sub_path = WORK_DIR / "submission_fcos.csv"
sub_df.to_csv(sub_path, index=False)
non_empty = (sub_df["PredictionString"].str.len() > 0).sum()
print(f"submission saved: {sub_path}")
print(f"non-empty: {non_empty}/{len(sub_df)} ({100*non_empty/len(sub_df):.1f}%)")

test frames: 3,811


test inference:   0%|          | 0/3811 [00:00<?, ?it/s]

submission saved: /kaggle/working/submission_fcos.csv
non-empty: 3811/3811 (100.0%)


In [12]:
print("=" * 70)
print("nb07b COMPLETE — FCOS family done")
print("=" * 70)
print()
print(f"Winner: {winner} (submitted via submission_fcos.csv)")
print(f"  mAP={winner_metrics['mAP']:.4f}  AP50={winner_metrics['AP50']:.4f}  "
      f"AP_small={winner_metrics['AP_small']:.4f}")
print()
print("Artifacts in /kaggle/working/:")
for p in sorted(WORK_DIR.glob("*fcos*")):
    print(f"  {p.name}  ({p.stat().st_size/1e6:.1f} MB)")
print()
print("NEXT STEPS:")
print("  1. Download submission_fcos.csv → upload to Kaggle LB")
print("  2. Upload F0+F1 artifacts as Kaggle dataset 'fcos-2'")
print("  3. Proceed to nb09 (RT-DETR query-based, eval-only)")
print("=" * 70)

run.finish()
print("W&B run finished:", run.url)

wandb: updating run metadata


nb07b COMPLETE — FCOS family done

Winner: F1 (submitted via submission_fcos.csv)
  mAP=0.0396  AP50=0.0837  AP_small=0.0201

Artifacts in /kaggle/working/:
  fcos_F0.pth  (128.9 MB)
  fcos_F1.pth  (128.9 MB)
  metrics_fcos_F0.json  (0.0 MB)
  metrics_fcos_F1.json  (0.0 MB)
  speed_fcos_F0.json  (0.0 MB)
  speed_fcos_F1.json  (0.0 MB)
  submission_fcos.csv  (60.2 MB)
  train_log_fcos_F1.json  (0.0 MB)
  val_preds_fcos_F0.json  (214.2 MB)
  val_preds_fcos_F1.json  (209.8 MB)

NEXT STEPS:
  1. Download submission_fcos.csv → upload to Kaggle LB
  2. Upload F0+F1 artifacts as Kaggle dataset 'fcos-2'
  3. Proceed to nb09 (RT-DETR query-based, eval-only)


wandb: uploading summary, console lines 157-182
wandb: 
wandb: Run history:
wandb:         speed/gpu_fps ▁
wandb:          speed/gpu_ms ▁
wandb:      train/epoch_loss ▁
wandb:    train/epoch_time_h ▁
wandb:            train/loss ▆▅▄▅▅▇▃█▄▄▅▇▃▅▅▅▄▄▃▂▅▃▆▄▄▇▃▄▃▅▃▄▃▃▃▅▂▄▂▁
wandb:        train/loss_box ▆▆█▆▅▅▇▅▄▃▆█▃▇▂▄▄▆▃▃▃█▆▄▄▄▁▄▃▄▃█▁▇▅▄▅▃▄▄
wandb: train/loss_centerness █▃▄▅▃▅▄▃▆▃▆█▂▄▆▃▇▇▃▄▆▄▆▃▆▅▅▆▅▃▂▁▇▇▄▄▂▅▅▆
wandb:        train/loss_cls ▄█▅█▇▆▄▇▅▄▃▅▅▆▅▆▇▇▆▇▃▄▂▄▃▅▅▄▄▅▁▅▂▆▃▄▁▄▆▅
wandb:            train/step ▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█████
wandb:              val/AP50 ▁
wandb:                    +5 ...
wandb: 
wandb: Run summary:
wandb:         speed/gpu_fps 3.34562
wandb:          speed/gpu_ms 298.89831
wandb:      train/epoch_loss 1.29734
wandb:    train/epoch_time_h 4.8159
wandb:            train/loss 1.3521
wandb:        train/loss_box 0.34755
wandb: train/loss_centerness 0.64445
wandb:        train/loss_cls 0.3601
wandb:            train/step 12080
wandb:              val/AP50

W&B run finished: https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det/runs/fx096sj7
